In [ ]:
import requests
import json
import datetime
import os
import csv

This notebook is used to collect education and AI related posts on bluesky

In [ ]:
# Both these lists should be changed as new terms become updated

AI_LIST = ['"Artificial Intelligence"',
           '"Chat Gpt"',
           'GenAI',
           '"Generative AI"',
           'gemeni',
           '"OpenAI"',
           'chatgpt4',
           'Chatbot',
           'magicschool',
           'AI',
           'GAI',
           '"magic school"']

EDU_LIST = ['Education',
            'Academia',
            'Classroom',
            'Teaching',
            'Teach',
            'research',
            'courses',
            '"Teaching tools"',
            '"Online learning"',
            '"Distance education"',
            '"Higher education"',
            '"k 12 education"',
            'Teachers',
            'schools',
            'student',
            'educators',
            'professor',
            '"personalized learning"',
            'test',
            'quiz',
            'assignment',
            'plagiarism',
            'cheating',
            'instructor',
            'college',
            'lesson',
            'edtech',
            'assessment',
            '"early childhood education"',
            'academic',
            '"k 8 education"',
            '"lesson plans"']

In [ ]:
# This code block collects all posts with a combination of any words from the lists
def get_post(query, filename, since='', until='') -> json:
  url = 'https://public.api.bsky.app/xrpc/app.bsky.feed.searchPosts?q='+query+'&since='+since+'Z&until='+until+'Z&lang=en&limit=100'
  payload={}
  headers = {
    'Accept': 'application/json',
    'Authorization': 'Bearer z7k5-egwu-wk7u-xxbx'
  }
  response = requests.request("GET", url, headers=headers, data=payload)
  with open(filename, 'w') as file:
     json.dump(response.json(), file, indent=2)
     file.close()

def get_multi_posts(start: datetime.datetime, end: datetime.datetime):
  '''Gets post in order from oldest to latest (beginning -> end)'''
  ai_query = ' | '.join(AI_LIST)
  edu_query = ' | '.join(EDU_LIST)
  query = '( '+ai_query+' ) .+ ( ' +edu_query +' )'
  while start < end:
     step = start + datetime.timedelta(hours=2)
     name = start.isoformat().replace(':', '_')
     get_post(query, 'jsons/'+name+'.json', since=start.isoformat(), until=step.isoformat())
     start = step

def get_multi_posts_reverse(start: datetime.datetime, end: datetime.datetime):
  '''Gets post in order from latest to oldest (wnd -> beginning)'''
  ai_query = ' | '.join(AI_LIST)
  edu_query = ' | '.join(EDU_LIST)
  query = '( '+ai_query+' ) .+ ( ' +edu_query +' )'
  while end > start:
     step = end - datetime.timedelta(hours=2)
     name = step.isoformat().replace(':', '_')  # removes protected characters (windows)
     get_post(query, 'jsons/'+name+'.json', since=step.isoformat(), until=end.isoformat())
     end = step

In [ ]:
#Last test ended 3/27/35 00:00:00
def main():
  gpt_release_UPDATED = datetime.datetime(2025, 3, 27, 0, 0, 0)
  until = datetime.datetime() # Fill this in with current date
  get_multi_posts_reverse(gpt_release_UPDATED, until)

main()

In [ ]:
# Removes any empty files from folder
def clean(filename) -> None:
    empty_file = '{\n  "posts": []\n}'
    with open(filename, 'r') as file:
        content = file.read()
        file.close()
    if content == empty_file:
        os.remove(filename)
        print(filename)

def get_files(dir) -> None:
    files = os.listdir(dir)
    for i in files:
        clean('jsons/'+i)

#get_files('jsons/')
print(os.listdir('jsons/'))

In [ ]:
# Creates a CSV of JSONS
def parse_to_csv(json_obj:str, file) -> None:
    with open(json_obj, 'r', encoding='utf-8') as x:
        full = json.loads(x.read())
        csv_writer = csv.writer(file)
        for posts in full['posts']:
            feature_list = [
                posts.get('uri', ''),
                posts['author'].get('displayName', ''),
                posts['record'].get('createdAt', ''),
                posts['author'].get('did', ''),
                posts['record'].get('text', '').replace('\n', ' '),
                posts['record'].get('reply', {}).get('parent', {}).get('uri', ''),
                'https://bsky.app/profile/' + posts['author'].get('handle', '') + '/post/' + posts.get('uri')[-13:]
            ]
            csv_writer.writerow(feature_list)

def iterate_folder(folder:str) -> None:
    all_files = os.listdir(folder)
    with open('bluesky_posts_all_FINAL.csv', 'w', newline='', encoding='utf-8') as file:
        for i in all_files:
            parse_to_csv(folder + i, file)
            print(i)

iterate_folder('jsons/')